In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations
from pyspark.sql.types import *
import pyspark.sql.functions as f

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Data availability check

In [0]:
source_path = data_paths["source"]["email"]

## Load source data

In [0]:
schema = StructType(
    [
        StructField("MBRSHP_NBR", StringType(), True),
        StructField("MAIL_ID", StringType(), True),
        StructField("EMAIL_SUBJECT", StringType(), True),
        StructField("EMAIL_NAME", StringType(), True),
        StructField("FIRST_BOUNCE_DATE", DateType(), True),
        StructField("LAST_BOUNCE_DATE", DateType(), True),
        StructField("BOUNCE_TOTAL", IntegerType(), True),
        StructField("FIRST_SEND_DATE", DateType(), True),
        StructField("LAST_SEND_DATE", DateType(), True),
        StructField("SEND_TOTAL", IntegerType(), True),
        StructField("FIRST_OPEN_DATE", DateType(), True),
        StructField("LAST_OPEN_DATE", DateType(), True),
        StructField("OPEN_TOTAL", IntegerType(), True),
        StructField("FIRST_CLICK_DATE", DateType(), True),
        StructField("LAST_CLICK_DATE", DateType(), True),
        StructField("CLICK_TOTAL", IntegerType(), True),
        StructField("FIRST_UNSUB_DATE", DateType(), True),
        StructField("LAST_UNSUB_DATE", DateType(), True),
        StructField("UNSUB_TOTAL", IntegerType(), True),
    ]
)

In [0]:
df = (
        spark.read.schema(schema)
        .option("header", "false")
        .option("sep", ",")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "true")
        .csv(source_path)
    )

In [0]:
df = df.repartition(f.col("FIRST_OPEN_DATE"))

In [0]:
validations.validate_table(
        spark, "source", 'email', config_validation, df, stats_etl_path
    )

In [0]:
df = df.where(f.length(f.col("MBRSHP_NBR")) == 11)
df.createOrReplaceTempView('df')

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_master_email}
    SELECT
        MBRSHP_NBR,
        MAIL_ID,
        EMAIL_SUBJECT,
        EMAIL_NAME,
        FIRST_BOUNCE_DATE,
        LAST_BOUNCE_DATE,
        BOUNCE_TOTAL,
        FIRST_SEND_DATE,
        LAST_SEND_DATE,
        SEND_TOTAL,
        FIRST_OPEN_DATE,
        LAST_OPEN_DATE,
        OPEN_TOTAL,
        FIRST_CLICK_DATE,
        LAST_CLICK_DATE,
        CLICK_TOTAL,
        FIRST_UNSUB_DATE,
        LAST_UNSUB_DATE,
        UNSUB_TOTAL
    FROM df
""")